# TriageNet Phase 1 Data Exploration

This notebook only loads `data/processed/cycles.parquet` and visualizes the unified cycle table.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
cycles_path = Path("..") / "data" / "processed" / "cycles.parquet"
cycles = pd.read_parquet(cycles_path)
cycles.info()

In [ ]:
summary = cycles.groupby("chemistry").agg(
    cells=("cell_id", "nunique"),
    cycles=("cycle_index", "count"),
    min_capacity_ah=("discharge_capacity_ah", "min"),
    max_capacity_ah=("discharge_capacity_ah", "max"),
    min_soh=("soh", "min"),
    max_soh=("soh", "max"),
)
summary

In [ ]:
grid = sns.relplot(
    data=cycles,
    x="cycle_index",
    y="soh",
    hue="cell_id",
    col="chemistry",
    col_wrap=2,
    kind="line",
    facet_kws={"sharex": False},
)
grid.set_axis_labels("Cycle index", "SOH")
grid.figure.suptitle("Capacity fade by cell", y=1.02)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
for chemistry, group in cycles.groupby("chemistry"):
    row = group.sort_values(["cell_id", "cycle_index"]).iloc[0]
    plt.plot(row["time_curve_s"], row["voltage_curve"], label=chemistry)
plt.xlabel("Time (s)")
plt.ylabel("Voltage (V)")
plt.title("Example discharge voltage curve per chemistry")
plt.legend()
plt.show()

In [ ]:
eol = cycles[cycles["is_eol_cycle"]]
sns.histplot(data=eol, x="soh", hue="chemistry", multiple="layer", bins=12)
plt.xlabel("EOL SOH")
plt.title("EOL SOH distribution by chemistry")
plt.show()

## What I observed

Fill this in after running the notebook on real ingested data. Mention cells with glitchy curves, chemistries with too few samples, missing datasets, and whether EOL SOH distributions look separable.

## Real-data observations

- Real ingestion currently contains 12 CALCE LCO cells and no LFP/NMC/NCA cells. This blocks Phase 2 because the chemistry classifier would have one class.
- CALCE CS2/CX2 cycles come from one institutional experiment, so manufacturer/source is fully confounded with chemistry until MIT or Sandia data is added.
- Some workbook-local cycles were skipped as junk/aborted runs because discharge capacity was <= 0.05 Ah; the loader logs each skip.
- Several late-life cycles have very low SOH. Phase 2 should decide whether to train on all cycles for EDA but use only EOL cycles for the triage input contract.
